# Week 4 Lab: Data Quality, Validation and Governance (Topic 4)
**Project:** Lapora, an MBRS XBRL tagging assistant that suggests SSMxT / MPERS concepts with citations and abstention.
**Input:** the Topic 3 hand-off (`prepared_chunks.jsonl` and `chunks.pkl`, 744 chunks from three SSM documents).
**Output:** a reviewed corpus (`prepared_chunks_clean.jsonl`, `chunks_clean.pkl`), validation/duplicate/quarantine logs and `docs/week4_data_quality_report.md`.

Principle for the whole lab: evidence, not a cosmetic pass rate. Where I did not check something, the report says **N.V. (Not Verified)**.

## Part A. Evidence-based quality assessment

### A0. Requirement, data need and quality dimension (Topic 2 link)
| Topic 2 requirement | What the data must support | Quality dimension | Evidence collected in this lab |
|---|---|---|---|
| **FR3** Every suggestion carries a citation (source, page, version) | Each chunk has a source file, a correct page and a document version | **Completeness**, **Consistency** | R2 provenance complete, R3 page in range, FAQ version-string check |
| **FR3 / NFR5** Citation must point at text that really exists | Chunk text matches the raw document at the stated page | **Accuracy** | 15-chunk spot check against the raw source (cell 6b) |
| **FR4** "Needs review" when evidence is weak (incl. the FAQ "Other" element rule) | Guidance text must be readable and whole; the "Other" rule must not be cut in half | **Completeness**, **Accuracy** | R5 short chunks, R6 mid-word cuts, S01-0073/0074 case |
| **FR4 / RAI1** Do not tag from outdated guidance | Documents must be the current SSM versions | **Timeliness** | Version strings only; newer-version check is N.V. |
| **FR3** One passage should not appear as several "different" citations | No repeated passages | **Uniqueness** | Exact-hash dedup (cell 7) |

### A1. Load the hand-off and verify it

In [1]:
# Cell 1 — Topic 4 setup and Topic 3 hand-off verification
try:
    import pandera.pandas as pa
    HAVE_PANDERA = True
except ImportError:
    try:
        !pip -q install pandera
        import pandera.pandas as pa
        HAVE_PANDERA = True
    except Exception:
        HAVE_PANDERA = False   # offline sandbox: the same rules run in plain pandas below
import os, re, json, pickle, hashlib
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path("/content")
for d in ("docs", "data"):
    (ROOT / d).mkdir(parents=True, exist_ok=True)
print("Pandera available:", HAVE_PANDERA)

with open(ROOT / "chunks.pkl", "rb") as f:
    chunks = pickle.load(f)
prepared_records = []
with open(ROOT / "prepared_chunks.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            prepared_records.append(json.loads(line))
print(f"Compatibility chunks: {len(chunks)}")
print(f"Inspectable JSONL records: {len(prepared_records)}")
if len(chunks) != len(prepared_records):
    raise ValueError("Topic 3 hand-off mismatch: JSONL and PKL chunk counts differ.")
same = all(c.page_content == r["text"] for c, r in zip(chunks, prepared_records))
print("PKL text identical to JSONL text, chunk by chunk:", same)
print("Topic 3 hand-off verified.")
print(chunks[0].page_content[:250])
print(chunks[0].metadata)

Pandera available: False
Compatibility chunks: 744
Inspectable JSONL records: 744
PKL text identical to JSONL text, chunk by chunk: True
Topic 3 hand-off verified.
MALAYSIAN BUSINESS REPORTING SYSTEM

(MBRS)

FREQUENTLY ASKED QUESTIONS
{'source': 'FAQ-MBRS.pdf', 'page': 1, 'source_id': 'S01', 'record_id': 'S01-0001', 'source_sha256': 'ef1fb2647ca1becc2491df33193294af7e242615bb5b6ff8f73a785f08d61c99', 'source_version': "Version 2.0 Jun 2022 (file title: update v2 31052022); 39 of 42 page headers read 'Version 1.0 Jun 2022'", 'is_toc': False, 'processing': 'paragraph-page-char-500'}


### A2. Quality inventory (missing, corrupted, noisy)

In [2]:
# Cell 2 — Build a chunk-quality inventory (Lapora fields added)
# Page counts per source come from the Topic 3 pipeline manifests (inspection evidence), not from new assumptions.
PIPE = ROOT / "week3_project" / "data" / "pipeline_v1"
PAGES = {}
for mf in PIPE.glob("*/pipeline_manifest.json"):
    m = json.loads(mf.read_text(encoding="utf-8"))
    PAGES[m["inspection"]["source"]] = m["inspection"]["pages"]
print("Pages per source (Topic 3 manifests):", PAGES)

def dotted_ratio(t):
    lines = [l for l in t.splitlines() if l.strip()]
    return sum(bool(re.search(r"\.{5,}", l)) for l in lines) / len(lines) if lines else 0.0

rows = []
for i, chunk in enumerate(chunks):
    text = chunk.page_content or ""
    meta = chunk.metadata or {}
    source = str(meta.get("source", "") or "")
    stripped = text.strip()
    rows.append({
        "chunk_id": i, "record_id": meta.get("record_id"), "source_id": meta.get("source_id"), "source": source,
        "page": meta.get("page"), "text": text, "char_count": len(stripped),
        "missing_text": len(stripped) == 0, "missing_source": source.strip() == "",
        "corrupted_char": ("\x00" in text) or ("�" in text),
        "noisy_whitespace": bool(re.search(r"\n{3,}| {4,}", text)),
        "dotted_ratio": dotted_ratio(text),
        "version_present": bool(str(meta.get("source_version", "")).strip()),
        "sha_present": len(str(meta.get("source_sha256", ""))) == 64,
        "starts_lower": stripped[:1].islower(),
    })
df = pd.DataFrame(rows)
# Mid-word boundary (V2): the previous chunk on the same page is a full 500-character slice and the
# boundary has an alphanumeric character on both sides. A paragraph boundary is stripped, so it cannot look like this.
prev_full = df["char_count"].shift(1).eq(500) & df["page"].shift(1).eq(df["page"]) & df["source_id"].shift(1).eq(df["source_id"])
prev_end_alnum = df["text"].shift(1).fillna("").str.rstrip().str[-1:].str.match(r"[A-Za-z0-9]")
this_start_alnum = df["text"].str.lstrip().str[:1].str.match(r"[a-z0-9]")
df["midword_start"] = prev_full & prev_end_alnum & this_start_alnum
df["midword_end"] = df["midword_start"].shift(-1, fill_value=False)
df["page_in_range"] = [isinstance(p, int) and 1 <= p <= PAGES.get(s, 0) for p, s in zip(df["page"], df["source"])]

print(df[["chunk_id", "source_id", "page", "char_count", "missing_text", "missing_source", "corrupted_char", "noisy_whitespace"]].head())
print(df["char_count"].describe())
df["is_toc_page"] = [bool(c.metadata.get("is_toc")) for c in chunks]
df["is_contents"] = df["is_toc_page"] | (df["dotted_ratio"] >= 0.4)
print("Missing text:", int(df["missing_text"].sum()))
print("Missing source metadata:", int(df["missing_source"].sum()))
print("Possible corrupted text:", int(df["corrupted_char"].sum()))
print("Possible noisy whitespace:", int(df["noisy_whitespace"].sum()))
print("Mid-word boundaries (chunks starting mid-word):", int(df["midword_start"].sum()), df.groupby("source_id")["midword_start"].sum().to_dict())
print("Page outside the source's page range:", int((~df["page_in_range"]).sum()))
print("Contents-page chunks (dotted-leader ratio >= 0.4):", int((df["dotted_ratio"] >= 0.4).sum()))

Pages per source (Topic 3 manifests): {'SSMxT2022_Architecture_Document.pdf.txt': 111, 'FAQ-MBRS.pdf': 43, 'User_manual_mTool_v2_FS2016.pdf.txt': 105}
   chunk_id source_id  page  ...  missing_source  corrupted_char  noisy_whitespace
0         0       S01     1  ...           False           False             False
1         1       S01     2  ...           False           False             False
2         2       S01     2  ...           False           False             False
3         3       S01     2  ...           False           False             False
4         4       S01     2  ...           False           False             False

[5 rows x 8 columns]
count    744.000000
mean     404.342742
std      144.966491
min        1.000000
25%      336.750000
50%      499.000000
75%      500.000000
max      500.000000
Name: char_count, dtype: float64
Missing text: 0
Missing source metadata: 0
Possible corrupted text: 0
Possible noisy whitespace: 0
Mid-word boundaries (chunks starting 

### A3. Manual inspection
Reading 10 random chunks (seed 42) plus every chunk under 40 characters, I recorded what I saw rather than what I expected.

In [3]:
# Cell 3 — Manual inspection (10 random chunks, seed 42) + two targeted samples
sample = df.sample(min(10, len(df)), random_state=42)
for idx in sample["chunk_id"]:
    c = chunks[int(idx)]
    print("\n--- CHUNK", idx, c.metadata.get("record_id"), "---")
    print("Source:", c.metadata.get("source", "?"), "| Page:", c.metadata.get("page", "?"), "| chars:", len(c.page_content))
    print(c.page_content[:320].replace("\n", " ⏎ "))


--- CHUNK 609 S03-0299 ---
Source: SSMxT2022_Architecture_Document.pdf.txt | Page: 74 | chars: 500
hich will usually have a negative value ⏎ or a positive value. Eg. Investment in subsidiaries will always have a positive value, ⏎ while expense items like Cost of sales will usually have a negative value. However, ⏎ there are no elements which should always be stored as a negative value in an instance ⏎ document as negatively

--- CHUNK 539 S03-0229 ---
Source: SSMxT2022_Architecture_Document.pdf.txt | Page: 57 | chars: 500
FS-EPC-1965; ⏎ FS-CLBG-1965; ⏎ FS-FC-1965; ⏎ FS-BNM-1965 ⏎ DisclosureOnWhetherCompan ⏎ yInvolvedAsManagementCom ⏎ panyUnderInterestSchemeAct ⏎ 2016 ⏎ DisclosureOnWhetherCompan ⏎ yInvolvedAsManagementComp ⏎ anyUnderInterestSchemeAct2 ⏎ 016 ⏎ Management company under ⏎ Interest Scheme Act 2016; ⏎ Not management company ⏎ under Interest Scheme Act ⏎ 201

--- CHUNK 694 S03-0384 ---
Source: SSMxT2022_Architecture_Document.pdf.txt | Page: 97 | chars: 120
which should be av

In [4]:
# Cell 3b — Targeted inspection: the shortest content chunks and the FAQ "Other" rule
short = df[(df["char_count"] < 40) & (df["dotted_ratio"] < 0.4)].sort_values("char_count")
print("Content chunks under 40 characters:", len(short))
print(short[["record_id", "source_id", "page", "char_count"]].assign(text=short["text"].str.replace("\n", " ⏎ ").str[:60]).to_string(index=False))
wrapped = df[~df["is_contents"]]["text"].str.contains(r"[A-Za-z]{18,}[ \t]*\n[ \t]*[a-z]{2,}")
print("\nContent chunks with a long identifier wrapped across lines (possible split element names):",
      int(wrapped.sum()), df[~df["is_contents"]][wrapped].groupby("source_id").size().to_dict())
for rid in ("S01-0073", "S01-0074"):
    r = df[df["record_id"] == rid].iloc[0]
    print("\n", rid, "page", r["page"], "chars", r["char_count"])
    print(r["text"][-260:] if rid.endswith("73") else r["text"][:260])

Content chunks under 40 characters: 20
record_id source_id  page  char_count                                   text
 S03-0231       S03    57           1                                      :
 S02-0131       S02    69           2                                     y.
 S02-0126       S02    67           2                                     he
 S03-0425       S03   108           3                                    the
 S03-0362       S03    91           4                                   bels
 S03-0051       S03    11           5                                  ns").
 S03-0144       S03    35           5                                  D.xml
 S02-0050       S02    26           9                              Figure 25
 S02-0162       S02    93           9                              template.
 S01-0002       S01     2          17                      Table of Contents
 S03-0169       S03    41          19                  172 ⏎ measurement 176
 S03-0326       S03    81          22

**Observation table**
| # | Observation | Where (record_id) | Looks like | Why it matters |
|---|---|---|---|---|
| 1 | Chunks begin or end in the middle of a word ("hich will usually…", "DisclosureOnWhetherCompan") | S03-0299, S03-0229 | Pipeline defect (500-character hard cut) | Retrieval may return a half-word that cannot be cited cleanly (FR3) |
| 2 | Single-token chunks: ":", "he", "y." | S03-0231, S02-0126, S02-0131 | Pipeline defect (fragment left by the cut) | Carry no meaning; would waste retrieval slots |
| 3 | Dotted-leader contents lines ("1. MBRS filing requirements ........") | S01-0002 onwards, pages 2-6 | Valid text, wrong for retrieval | Contents pages match every keyword but answer nothing |
| 4 | The FAQ "Other" rule (2.3.1.4, page 25) is split across two chunks | S01-0073 / S01-0074 | Pipeline defect | The FR4 abstention rule loses half its evidence |
| 5 | Long SSMxT element names wrapped over lines and split ("DisclosureOnWhetherCompan / yInvolved…") | 53 S03 content chunks | Source layout plus pipeline cut | An element name that is split cannot be matched to a concept |
| 6 | Table cell "d) Others (if any) PDF" | S01-0126 | Valid exception | Short but genuine content |

**Reflection.** What surprised me: the text is clean (no missing or corrupted characters, no noisy whitespace), so the real problem is *structural*, not character-level. A simple "no nulls" check would have given a 100% pass and hidden the problem that matters most for citations: chunk boundaries.

### A4. Validation rules (condition, why, failures, decision)

In [5]:
# Cell 4 — Repeatable validation rules (pandas rules are the evidence; Pandera runs the structural subset when installed)

RULES = [
    # id, name, failure it detects, condition (True = failure), decision
    ("R1", "Text present", "Empty chunk gives no evidence", lambda d: d["missing_text"], "Hard fail"),
    ("R2", "Provenance complete (source, version, 64-char SHA-256)", "Untraceable evidence cannot be cited (FR3)",
     lambda d: d["missing_source"] | ~d["version_present"] | ~d["sha_present"], "Hard fail"),
    ("R3", "Page is an integer inside the source's page range", "Citation points to a page that does not exist (FR3)",
     lambda d: ~d["page_in_range"], "Hard fail"),
    ("R4", "No corrupted characters (NUL, U+FFFD)", "Extraction damage", lambda d: d["corrupted_char"], "Hard fail"),
    ("R5", "Content chunk >= 40 characters", "Fragment or stray label with no standalone meaning",
     lambda d: (d["char_count"] < 40) & ~d["is_contents"], "Review"),
    ("R6", "No word cut at a chunk boundary", "Rule text split mid-word, so a retrieved half is incomplete (FR4, RAI1)",
     lambda d: d["midword_start"], "Hard fail (pipeline defect)"),
    ("R7", "Not a table-of-contents chunk", "Dotted-leader navigation text would be retrieved as evidence",
     lambda d: d["is_contents"], "Quarantine"),
    ("R8", "No noisy whitespace (4+ spaces or 3+ blank lines)", "Layout noise", lambda d: d["noisy_whitespace"], "Warning"),
]
fail_rows, summary = [], []
for rid, name, why, cond, decision in RULES:
    mask = cond(df)
    for _, r in df[mask].iterrows():
        fail_rows.append({"rule_id": rid, "rule": name, "chunk_id": r["chunk_id"], "record_id": r["record_id"],
                          "source_id": r["source_id"], "page": r["page"], "char_count": r["char_count"],
                          "decision": decision, "text_preview": r["text"][:60].replace("\n", " ")})
    summary.append({"rule": rid, "condition": name, "failures": int(mask.sum()),
                    "percent_of_744": round(100 * mask.sum() / len(df), 1), "decision": decision})
failure_cases = pd.DataFrame(fail_rows)
failure_cases.to_csv(ROOT / "docs" / "week4_validation_failures.csv", index=False)
print(pd.DataFrame(summary).to_string(index=False))
print("Failure rows saved:", len(failure_cases), "-> docs/week4_validation_failures.csv")
print("Failures by rule and source:")
print(failure_cases.groupby(["rule_id", "source_id"]).size().unstack(fill_value=0))

if HAVE_PANDERA:
    schema = pa.DataFrameSchema({
        "chunk_id": pa.Column(int, checks=pa.Check.ge(0), nullable=False),
        "source": pa.Column(str, checks=pa.Check(lambda s: s.str.strip().str.len() > 0, error="source metadata must not be empty")),
        "text": pa.Column(str, checks=pa.Check(lambda s: s.str.strip().str.len() > 0, error="chunk text must not be empty")),
        "corrupted_char": pa.Column(bool, checks=pa.Check(lambda s: ~s, error="corrupted characters found")),
        "page_in_range": pa.Column(bool, checks=pa.Check(lambda s: s, error="page outside source page range")),
    }, strict=False)
    try:
        schema.validate(df, lazy=True); print("Pandera structural schema: all rules passed.")
    except pa.errors.SchemaErrors as exc:
        print("Pandera failures:", len(exc.failure_cases))
else:
    print("Pandera is not installed in this environment: R1-R4 (the structural rules) were run in pandas instead.")

rule                                              condition  failures  percent_of_744                    decision
  R1                                           Text present         0             0.0                   Hard fail
  R2 Provenance complete (source, version, 64-char SHA-256)         0             0.0                   Hard fail
  R3      Page is an integer inside the source's page range         0             0.0                   Hard fail
  R4                  No corrupted characters (NUL, U+FFFD)         0             0.0                   Hard fail
  R5                         Content chunk >= 40 characters        19             2.6                      Review
  R6                        No word cut at a chunk boundary       258            34.7 Hard fail (pipeline defect)
  R7                          Not a table-of-contents chunk        66             8.9                  Quarantine
  R8      No noisy whitespace (4+ spaces or 3+ blank lines)         0             0.0   

In [6]:
# Cell 5 — Document-level consistency and timeliness (needs the raw sources from Topic 3)
RAW = ROOT / "week3_project" / "data" / "raw"
faq_pdf, mtool_txt, arch_txt = RAW / "FAQ-MBRS.pdf", RAW / "User_manual_mTool_v2_FS2016.pdf.txt", RAW / "SSMxT2022_Architecture_Document.pdf.txt"
if faq_pdf.exists():
    from pypdf import PdfReader
    rd = PdfReader(str(faq_pdf))
    print("FAQ metadata title:", (rd.metadata or {}).get("/Title"))
    ver = Counter()
    for p in rd.pages:
        for m in re.finditer(r"Version\s+(\d+\.\d+)\s+(\w+\s+\d{4})", p.extract_text() or ""):
            ver[m.group(1) + " " + m.group(2)] += 1
    print("FAQ version strings across pages:", dict(ver))
else:
    print("FAQ raw file not available: N.V.")
if mtool_txt.exists():
    t = mtool_txt.read_text(encoding="utf-8")
    print("mTool manual version strings:", dict(Counter(re.findall(r"Version\s+\d+\.\d+\s+\w+\s+\d{4}", t))))
if arch_txt.exists():
    t = arch_txt.read_text(encoding="utf-8")
    print("Architecture document taxonomy versions:", dict(Counter(re.findall(r"SSMxT_2022\s?v\d+\.\d+", t))))
    print("Architecture document: 4-digit years mentioned (top):", Counter(re.findall(r"\b(20[12]\d)\b", t)).most_common(5))

FAQ metadata title: FAQ-MBRS (BI) - update v2 31052022
FAQ version strings across pages: {'2.0 Jun 2022': 3, '1.0 Jun 2022': 39}
mTool manual version strings: {'Version 2.4 September 2024': 104}
Architecture document taxonomy versions: {'SSMxT_2022v1.0': 99, 'SSMxT_2022\nv1.0': 1}
Architecture document: 4-digit years mentioned (top): [('2016', 97), ('2022', 23), ('2014', 2), ('2017', 1), ('2015', 1)]


**Rules table (real counts from this run)**
| Rule | Condition | Why it matters | Failures / 744 | Decision |
|---|---|---|---|---|
| R1 | Text present | Empty chunk cannot be cited | 0 | Hard fail |
| R2 | Source, version and 64-char SHA-256 present | FR3 citation needs provenance | 0 | Hard fail |
| R3 | Page is an integer inside the source's page range | Wrong page = wrong citation | 0 | Hard fail |
| R4 | No NUL / U+FFFD characters | Corruption breaks matching | 0 | Hard fail |
| R5 | Content chunk of at least 40 characters | Very short chunks carry no evidence | 19 (S01 1, S02 6, S03 12) | Review |
| R6 | Chunk does not start/end by cutting a word | Half-words cannot be cited or matched | 258 (S01 11, S02 49, S03 198) | Hard fail (pipeline defect), kept but flagged |
| R7 | Not a table-of-contents chunk | Contents pages pollute retrieval | 66 (S01 21, S02 19, S03 26) | Quarantine |
| R8 | No long runs of blank lines/spaces | Layout noise | 0 | Warning |

Validation detects; engineering judgement decides. R6 is a *hard fail* by rule, but I did not delete 258 chunks (35% of the corpus): I flagged them and recorded the real fix (re-chunk, see report section 6). Pandera is not installed in this sandbox, so the structural rules R1-R4 ran in pandas; the Pandera block runs when the package is available (Colab).

### A5. Triage of failed cases

In [7]:
# Cell 6 — Triage: inspect evidence for five flagged cases
def ctx(rid, before=70, after=70):
    i = int(df.index[df["record_id"] == rid][0])
    prev_t = chunks[i - 1].page_content if i > 0 else ""
    next_t = chunks[i + 1].page_content if i + 1 < len(chunks) else ""
    cur = chunks[i].page_content
    return prev_t[-before:].replace("\n", " ⏎ "), cur[:160].replace("\n", " ⏎ "), next_t[:after].replace("\n", " ⏎ ")
for rid in ("S02-0126", "S03-0231", "S01-0126", "S01-0002", "S01-0073"):
    p, c, n = ctx(rid)
    print(f"\n{rid}  (rules: {sorted(failure_cases.loc[failure_cases.record_id == rid, 'rule_id'])})")
    print("  previous chunk ends :", repr(p)); print("  this chunk          :", repr(c)); print("  next chunk starts   :", repr(n))
# Verify the quarantine rule both ways
print("\nContents-chunk pages by source (Topic 3 recorded: S01 2-4, S02 2-5, S03 3-6):")
print(df[df["is_contents"]].groupby("source_id")["page"].agg(["min", "max", "count"]).to_string())
borderline = df[(~df["is_contents"]) & (df["dotted_ratio"] > 0)]
print("Non-contents chunks with any dotted leader line (possible false negatives):", len(borderline), borderline["record_id"].tolist()[:5])
# raw-page evidence for S01-0126 (a genuine short table cell?)
if faq_pdf.exists():
    row = df[df["record_id"] == "S01-0126"].iloc[0]
    raw_page = rd.pages[int(row["page"]) - 1].extract_text()
    k = raw_page.find("Others (if any)")
    print("\nS01-0126 raw page", int(row["page"]), "context:", repr(raw_page[max(0, k - 160): k + 40]))


S02-0126  (rules: ['R5', 'R6'])
  previous chunk ends : "d in ⏎ Financial Statement, the 'Type Domain > Add' button is to allow t"
  this chunk          : 'he'
  next chunk starts   : 'user to add the required number of new rows to capture the data ⏎ regard'

S03-0231  (rules: ['R5'])
  previous chunk ends : 'nformation can be found at SSM portal website using the ⏎ following link'
  this chunk          : ':'
  next chunk starts   : 'https://mbrs.ssm.com.my/mbrs/groupAction?grpmn=D8wxlh8uNepxxI8JsSa0Hw='

S01-0126  (rules: ['R5'])
  previous chunk ends : 'b) Application letter from ⏎ director ⏎ c) Unaudited Financial ⏎ Statements ⏎ '
  this chunk          : 'd) Others (if any) ⏎ PDF'
  next chunk starts   : '9 Application for relief from ⏎ requirements as to form ⏎ and contents of '

S01-0002  (rules: ['R7'])
  previous chunk ends : 'ALAYSIAN BUSINESS REPORTING SYSTEM ⏎  ⏎ (MBRS) ⏎  ⏎ FREQUENTLY ASKED QUESTIONS'
  this chunk          : 'Table of Contents'
  next chunk starts   : '1. M

In [8]:
# Cell 6b — Accuracy spot check: do 15 sampled content chunks appear in the raw source (and on the stated page)?
def norm(x): return re.sub(r"\s+", "", x)
content = df[~df["is_contents"]]
samp = pd.concat([content[content.source_id == s_].sample(5, random_state=42) for s_ in ("S01", "S02", "S03")])
page_txt = {int(i + 1): norm(p.extract_text() or "") for i, p in enumerate(rd.pages)} if faq_pdf.exists() else {}
raw_txt = {"S02": norm(mtool_txt.read_text(encoding="utf-8")) if mtool_txt.exists() else "", "S03": norm(arch_txt.read_text(encoding="utf-8")) if arch_txt.exists() else ""}
ok = []
for _, r in samp.iterrows():
    body = [norm(l) for l in r["text"].splitlines() if l.strip()]
    if r["source_id"] == "S01":
        found = all(l in page_txt.get(int(r["page"]), "") for l in body)
    else:
        found = norm(r["text"]) in raw_txt[r["source_id"]] or all(l in raw_txt[r["source_id"]] for l in body)
    ok.append(found)
samp = samp.assign(text_found_in_source=ok)
print(samp[["record_id", "page", "char_count", "text_found_in_source"]].to_string(index=False))
print("Sampled chunks whose text is found in the raw source:", int(sum(ok)), "of", len(ok), "(S01 also checked on the stated page)")

record_id  page  char_count  text_found_in_source
 S01-0102    33         393                  True
 S01-0026     6         492                  True
 S01-0062    20         162                  True
 S01-0091    29         369                  True
 S01-0032     8         492                  True
 S02-0146    83         207                  True
 S02-0065    33         500                  True
 S02-0153    86         129                  True
 S02-0158    91         500                  True
 S02-0131    69           2                  True
 S03-0097    23         499                  True
 S03-0245    60         500                  True
 S03-0285    69         123                  True
 S03-0060    14         242                  True
 S03-0069    17         499                  True
Sampled chunks whose text is found in the raw source: 15 of 15 (S01 also checked on the stated page)


| Case | Rule | Class | Reasoning | Action |
|---|---|---|---|---|
| S02-0126 ("he", p.67) | R5, R6 | **Defect** | A leftover fragment from a 500-character cut | Retain, flag; re-chunk later |
| S03-0231 (":", p.57) | R5, R6 | **Defect** | Single punctuation mark with no content | Retain, flag (review); re-chunk later |
| S01-0126 ("d) Others (if any) PDF", p.40) | R5 | **Valid exception** | Raw page 40 shows it is a real table cell | Retain |
| S01-0002 ("Table of Contents", p.2) | R7 | **Defect for retrieval** | Contents heading, no guidance | **Quarantine** |
| S01-0073 / 0074 (the "Other" rule, p.25) | R6 | **Defect, high impact** | The FR4 rule is split; each half is incomplete | Retain, flag; **re-chunk before Topic 7** |

Decisions are explicit: retain / correct / remove / quarantine / review. "Correct" (re-chunking) is not done in this lab because it changes the Topic 3 pipeline; it is a condition in the quality gate.

## Part B. Duplicates, coverage, bias and governance

### B1. Exact-hash duplicate detection with provenance
Chunk text is normalised (whitespace and case) and hashed with SHA-256. Every duplicate would be logged with the record it duplicates.

In [9]:
# Cell 7 — Exact-hash deduplication with provenance preserved (same method as the lab)
seen, duplicate_records = {}, []
for i, chunk in enumerate(chunks):
    normalized = re.sub(r"\s+", " ", chunk.page_content.strip())
    h = hashlib.sha256(normalized.encode("utf-8")).hexdigest()
    rid = chunk.metadata.get("record_id")
    if h not in seen:
        seen[h] = {"first_record": rid, "first_source": chunk.metadata.get("source", "?")}
    else:
        duplicate_records.append({"hash": h, "duplicate_record": rid, "duplicate_source": chunk.metadata.get("source", "?"),
                                  "first_record": seen[h]["first_record"], "first_source": seen[h]["first_source"], "preview": normalized[:160]})
print("Before:", len(chunks)); print("Distinct normalised texts:", len(seen)); print("Exact duplicates found:", len(duplicate_records))
dup_df = pd.DataFrame(duplicate_records, columns=["hash", "duplicate_record", "duplicate_source", "first_record", "first_source", "preview"])
dup_df.to_csv(ROOT / "docs" / "week4_duplicate_review.csv", index=False)
# Near-duplicate signal (not proof): same first 100 characters
prefix = Counter(re.sub(r"\s+", " ", c.page_content.strip())[:100] for c in chunks if len(c.page_content.strip()) >= 100)
near = {k: v for k, v in prefix.items() if v > 1}
print("Chunks sharing an identical 100-character opening (near-duplicate signal):", sum(near.values()), "in", len(near), "groups")
for k, v in list(near.items())[:3]: print("  x%d" % v, repr(k[:80]))

Before: 744
Distinct normalised texts: 744
Exact duplicates found: 0
Chunks sharing an identical 100-character opening (near-duplicate signal): 0 in 0 groups


**Result:** 0 exact duplicates and 0 chunks sharing a 100-character opening. This does **not** show the corpus is semantically unique: exact hashing cannot see paraphrases, and the FAQ and the mTool manual do describe the same filing steps in different words. `docs/week4_duplicate_review.csv` is written (empty, header only) so the check is repeatable when the corpus changes.

### Quarantine, reviewed corpus and reload check

In [10]:
# Cell 8 — Build the reviewed corpus: quarantine contents chunks, keep flagged-but-usable chunks with quality_flags
dup_records = set(dup_df["duplicate_record"])
quarantine_mask = df["is_contents"] | df["record_id"].isin(dup_records)
print("Quarantined:", int(quarantine_mask.sum()), "(contents chunks:", int(df["is_contents"].sum()), "| exact duplicates:", len(dup_records), ")")
q_rows = df[quarantine_mask][["chunk_id", "record_id", "source_id", "page", "char_count"]].assign(reason="table-of-contents chunk (navigation text)")
q_rows.to_csv(ROOT / "docs" / "week4_quarantine.csv", index=False)

clean_chunks, clean_records = [], []
for i, chunk in enumerate(chunks):
    if quarantine_mask.iloc[i]:
        continue
    r = df.iloc[i]
    flags = [f for f, on in (("midword_start", r["midword_start"]), ("midword_end", r["midword_end"]),
                             ("short_under_40", r["char_count"] < 40), ("noisy_whitespace", r["noisy_whitespace"]),
                             ("contact_detail", bool(re.search(r"[\w.+-]+@[\w-]+\.[\w.]+|(?<!\d)(?:\+?60[- ]?\d{1,2}|0\d{1,2})[- ]\d{3,4}[- ]\d{4}", chunk.page_content)))) if on]
    meta = dict(chunk.metadata); meta["quality_flags"] = flags; meta["topic3_chunk_id"] = int(i)
    chunk2 = type(chunk)(page_content=chunk.page_content, metadata=meta)
    clean_chunks.append(chunk2)
    clean_records.append({"chunk_id": len(clean_records), "text": chunk.page_content, "metadata": meta})
with open(ROOT / "data" / "chunks_clean.pkl", "wb") as f: pickle.dump(clean_chunks, f)
with open(ROOT / "data" / "prepared_chunks_clean.jsonl", "w", encoding="utf-8") as f:
    for r in clean_records: f.write(json.dumps(r, ensure_ascii=False) + "\n")

# Reload both and confirm they agree
with open(ROOT / "data" / "chunks_clean.pkl", "rb") as f: re_pkl = pickle.load(f)
re_jsonl = [json.loads(l) for l in open(ROOT / "data" / "prepared_chunks_clean.jsonl", encoding="utf-8") if l.strip()]
assert len(re_pkl) == len(re_jsonl) == len(clean_chunks), "clean PKL and JSONL counts differ"
assert all(a.page_content == b["text"] for a, b in zip(re_pkl, re_jsonl)), "clean PKL and JSONL text differ"
print("Reviewed corpus:", len(re_pkl), "chunks (JSONL and PKL agree)")
print("Chunks carrying at least one quality flag:", sum(1 for c in re_pkl if c.metadata["quality_flags"]))
print(Counter(f for c in re_pkl for f in c.metadata["quality_flags"]))

Quarantined: 66 (contents chunks: 66 | exact duplicates: 0 )
Reviewed corpus: 678 chunks (JSONL and PKL agree)
Chunks carrying at least one quality flag: 415
Counter({'midword_end': 244, 'midword_start': 244, 'short_under_40': 19, 'contact_detail': 3})


### B2. Coverage versus application scope

In [11]:
# Cell 9 — Coverage by source and by application area (keyword PROXY: no topic metadata was captured in Topic 3)
clean_df = pd.DataFrame([{"source_id": c.metadata["source_id"], "text": c.page_content, "flags": c.metadata["quality_flags"]} for c in re_pkl])
src_cov = clean_df.groupby("source_id").size().rename("chunks").reset_index()
src_cov["percent"] = (100 * src_cov["chunks"] / len(clean_df)).round(1)
print(src_cov.to_string(index=False))
AREAS = {
    "SOFP / financial position": r"(?i:financial\s+position|balance\s+sheet)|\bSOFP\b",
    "P&L / profit or loss": r"(?i:profit\s+or\s+loss|income\s+statement)",
    "'Other' element guidance": r"[“\"]Other[”\"]|\bOther (?:element|category|section)",
    "MPERS": r"\bMPERS\b", "MFRS": r"\bMFRS\b", "Auto Tag": r"(?i:auto\s?tag)",
}
area = pd.DataFrame({a: clean_df["text"].str.contains(rx, regex=True).groupby(clean_df["source_id"]).sum() for a, rx in AREAS.items()}).astype(int)
area.loc["Total"] = area.sum()
print("\nChunks matching each application area (keyword proxy):")
print(area.to_string())
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
ax[0].bar(src_cov["source_id"], src_cov["chunks"], color=["#1F7A6D", "#7A9E9F", "#E0A526"][:len(src_cov)])
for x, v, p in zip(src_cov["source_id"], src_cov["chunks"], src_cov["percent"]): ax[0].text(x, v + 4, f"{v} ({p}%)", ha="center")
ax[0].set_title("Reviewed corpus: chunks by source"); ax[0].set_ylabel("Number of chunks")
area.drop("Total").T.plot(kind="barh", stacked=True, ax=ax[1], color=["#1F7A6D", "#7A9E9F", "#E0A526"])
ax[1].set_title("Keyword proxy: chunks per application area"); ax[1].set_xlabel("Chunks (a signal, not proof of adequacy)")
plt.tight_layout(); plt.savefig(ROOT / "docs" / "week4_coverage_by_source.png", dpi=150); plt.close()
print("Saved docs/week4_coverage_by_source.png")

source_id  chunks  percent
      S01     114     16.8
      S02     157     23.2
      S03     407     60.0

Chunks matching each application area (keyword proxy):
           SOFP / financial position  P&L / profit or loss  'Other' element guidance  MPERS  MFRS  Auto Tag
source_id                                                                                                  
S01                                2                     3                         2      9     9         1
S02                                4                     3                         0      3     4        21
S03                                8                     3                         0     21    24         0
Total                             14                     9                         2     33    37        22
Saved docs/week4_coverage_by_source.png


**Expected vs actual evidence**
| Application need (Lapora scope) | Expected evidence | Actual (keyword proxy, reviewed corpus) | Reading |
|---|---|---|---|
| Statement of financial position tagging | Substantial (core MBRS statement) | 14 chunks (S01 2, S02 4, S03 8) | Present, thin for a core statement |
| Profit or loss tagging | Substantial | 9 chunks | Thin |
| FAQ "Other" element rule | At least the rule, whole | 2 chunks, both from S01, split in half | Present but damaged (S01-0073/74) |
| MPERS guidance | Present (SMEs are the main users) | 33 chunks (S01 9, S02 3, S03 21) | Covered mostly by the architecture document |
| MFRS guidance | Present | 37 chunks | Covered |
| mTool Auto Tag workflow | Present | 22 chunks, almost all S02 | Covered by the manual only |

**Reflection.** The corpus is 60.0% architecture document (S03, 407 chunks), 23.2% mTool manual (S02, 157) and 16.8% FAQ (S01, 114). Unequal chunk counts are a signal, not proof: S03 is the longest document and its element tables are split into many small chunks. What the chart *cannot* tell me is whether the guidance is adequate, because the keyword proxy counts mentions, not coverage of every concept. A real concept list (the SSMxT taxonomy itself) is not in the corpus yet, which matters for FR2. Keyword counts are a proxy only.

### B3. Bias and representation risks

In [12]:
# Cell 10 — Bias-risk evidence (data-level signals only; not a fairness claim)
by_src = pd.DataFrame({
    "content_chunks": clean_df.groupby("source_id").size(),
    "mid_word_start": df[~df["is_contents"]].groupby("source_id")["midword_start"].sum(),
    "short_under_40": df[~df["is_contents"]].groupby("source_id").apply(lambda d: int((d["char_count"] < 40).sum())),
    "full_500_char_chunks": df[~df["is_contents"]].groupby("source_id").apply(lambda d: int((d["char_count"] == 500).sum())),
})
by_src["mid_word_rate_%"] = (100 * by_src["mid_word_start"] / by_src["content_chunks"]).round(1)
by_src["share_of_corpus_%"] = (100 * by_src["content_chunks"] / by_src["content_chunks"].sum()).round(1)
print(by_src.to_string())
mp = clean_df["text"].str.contains(r"\bMPERS\b").groupby(clean_df["source_id"]).sum(); mf = clean_df["text"].str.contains(r"\bMFRS\b").groupby(clean_df["source_id"]).sum()
print("\nChunks mentioning MPERS:", mp.to_dict(), "| mentioning MFRS:", mf.to_dict())

           content_chunks  mid_word_start  short_under_40  full_500_char_chunks  mid_word_rate_%  share_of_corpus_%
source_id                                                                                                          
S01                   114               8               1                    10              7.0               16.8
S02                   157              47               6                    48             29.9               23.2
S03                   407             189              12                   243             46.4               60.0

Chunks mentioning MPERS: {'S01': 9, 'S02': 3, 'S03': 21} | mentioning MFRS: {'S01': 9, 'S02': 4, 'S03': 24}


These are **potential representation risks**, not fairness claims (fairness evaluation is Topic 8).
| Risk | Type | Propagation path (Source / Pipeline decision → Prepared corpus → Later retrieval → User-facing effect) | Mitigation | Test |
|---|---|---|---|---|
| Corpus is 60% one document (S03) | Representation | Three-source selection → S03 dominates chunks → retrieval returns S03 passages for most queries → user sees architecture-style answers, fewer plain FAQ explanations | Retrieval cap per source or re-weighting; chunk by section not by 500 characters | Run 20 FAQ-style questions; count the share of top-5 hits from each source |
| Mid-word cuts are 46.4% of S03, 29.9% of S02 and 7.0% of S01 | Measurement (pipeline) | Fixed 500-character cut → S03/S02 chunks damaged more often → those chunks rank worse or cite half-words → users see fewer or poorer citations for S02/S03 topics | Sentence-aware chunking with overlap; keep the flag until fixed | Re-run R6 per source; target under 5% each |
| Guidance reflects the 2022 / 2024 SSM view of MFRS and MPERS only | Historical | Older document versions → prepared corpus → retrieval of outdated guidance → user may tag against superseded rules | Version check against newer SSM publications; show version in every citation (already in FR3) | Compare document versions with the current SSM site (N.V. until I can reach it) |

### B4. Governance record

In [13]:
# Cell 11 — Privacy / governance scan of the reviewed corpus and the provenance record
all_text = "\n".join(c.page_content for c in re_pkl)
emails = sorted(set(re.findall(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+", all_text)))
phones = sorted(set(re.findall(r"(?<!\d)(?:\+?60[- ]?\d{1,2}[- ]?\d{3,4}[- ]?\d{4}|0\d{1,2}[- ]\d{3,4}[- ]\d{4})(?!\d)", all_text)))
nric = re.findall(r"\b\d{6}-\d{2}-\d{4}\b", all_text)
print("Email addresses:", emails); print("Phone-like numbers:", phones); print("NRIC-like numbers:", len(nric))
man = json.loads((ROOT / "week3_project" / "data" / "week3_manifest.json").read_text(encoding="utf-8"))["records"]
for m in man:
    print(m["source_id"], "| owner:", m["source_owner"], "| url present:", bool(m["source_url"]), "| rights:", m["access_rights"][:70])

Email addresses: ['customercare@digicert.com.my', 'enquiry@ssm.com.my', 'refund@ssm.com.my']
Phone-like numbers: ['+603 8800 8008', '03-7721 4000', '03-7721 4001']
NRIC-like numbers: 0
S01 | owner: Suruhanjaya Syarikat Malaysia (SSM) | url present: True | rights: Public SSM document (URL). Redistribution terms not verified: raw file
S02 | owner: Suruhanjaya Syarikat Malaysia (SSM) | url present: True | rights: Public SSM document (URL). This is a text conversion supplied by the s
S03 | owner: Suruhanjaya Syarikat Malaysia (SSM) | url present: True | rights: Public SSM document (URL). This is a text conversion supplied by the s


| Item | S01 FAQ-MBRS | S02 mTool manual | S03 SSMxT_2022 architecture |
|---|---|---|---|
| Owner / version / provenance | SSM; FAQ conflicting version strings (39 pages "1.0 Jun 2022", 3 pages "2.0 Jun 2022"; PDF title "update v2 31052022"); SHA-256 recorded | SSM; v2.4 September 2024; text conversion; SHA-256 recorded | SSM; SSMxT_2022 v1.0; no document date; text conversion; SHA-256 recorded |
| Rights / access | Public document. **Redistribution terms not verified (N.V.)** | Same | Same |
| Personal data | 3 organisational emails, 3 phone numbers, 0 NRIC-like numbers (corpus total) | included in the same scan | included in the same scan |
| Consent basis | Not applicable: no individuals' data, organisational contact details only | Not applicable | Not applicable |
| Update / retention | Re-validate when SSM publishes a new version or when chunking changes; keep the raw files out of Git | Same | Same |

Publicly accessible does not mean public domain. The lab asks me to commit the chunk files, but they contain SSM text. I treat this as an open rights issue: the report records it, and the files can be removed from Git (or the repository made private) if SSM terms do not allow it.

## Part C. Report and commit
The report is `docs/week4_data_quality_report.md` (8 sections). Artefacts committed: this notebook, `prepared_chunks_clean.jsonl`, `chunks_clean.pkl`, `week4_validation_failures.csv`, `week4_duplicate_review.csv`, `week4_quarantine.csv`, `week4_coverage_by_source.png`, plus the Topic 3 hand-off. The clean files were **reloaded and verified** in cell 8 (JSONL and PKL counts and text agree) before committing.